# Notebook 3: Cyclobutene Ring Opening — A Reaction Where Single-Reference Fails

**CSC Spring School on Computational Chemistry 2026**  
**Multireference Quantum Chemistry**

---

## Where we are coming from

In Notebook 1 you saw that stretching the H₂ bond eventually forces the wavefunction into a two-configuration description — static correlation becomes essential and single-reference methods fail. That was the simplest possible bond, chosen for transparency.

This notebook looks at a real organic reaction: the **electrocyclic ring opening of cyclobutene to butadiene**. The breaking bond is a C–C single bond, and the reaction passes through a transition state with significant diradical character. The same physics applies — but now in a chemically relevant context.

The disrotatory pathway (thermally forbidden by the Woodward-Hoffmann rules) has an experimental barrier of approximately 40 kcal/mol. We will compute the reaction path with three methods and see how they compare.

---

## Learning objectives

By the end of this notebook you will be able to:

- Identify the active space for an electrocyclic reaction
- Explain why a single-reference method underestimates the barrier at a diradical TS
- Interpret NO occupation numbers along a reaction path
- Describe what dynamic correlation (NEVPT2) adds on top of CASSCF

---

## Setup

In [ ]:
import re
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

from utils import (
    ORCA, HARTREE_TO_KCAL,
    setup_workdir, clean_workdir,
    run_orca, get_energy, get_casscf_energy, get_nevpt2_energy,
    get_no_occupations, terminated_normally,
    parse_xyz, bond_length, stretch_bond, atoms_to_xyz_block,
)

NPROCS = 4
print(f"ORCA: {ORCA}")
print(f"NPROCS: {NPROCS}")

---
## Working directory

All ORCA input and output files will be written to the `cyc/` subdirectory to keep the notebook directory clean. Run the cell below at the start of each session.

If you see a warning about stale files from a previous run, run `clean_workdir('cyc')` before proceeding — stale `.gbw` files can cause CASSCF to read wrong starting orbitals and converge to an incorrect state.

In [ ]:
work_dir = setup_workdir('cyc')

In [ ]:
# Run this cell only if setup_workdir() warned about stale files above
# clean_workdir('cyc')
# work_dir = setup_workdir('cyc')

---
## The molecule

Cyclobutene is a four-membered ring with one C=C double bond. The ring opening breaks the C–C single bond opposite the double bond, forming the conjugated diene butadiene.

Run the cell below to write the starting geometry and visualise it.

In [ ]:
xyz_text = """10

C          0.31220        1.03779       -0.03371
C          1.04130       -0.31901        0.03561
C         -0.33386       -0.94725        0.00421
C         -0.97076        0.23765       -0.00426
H          0.50055        1.60693       -0.95079
H          0.49235        1.69357        0.82501
H          1.61890       -0.47668        0.95330
H          1.68848       -0.53012       -0.82239
H         -0.66756       -1.98084        0.01270
H         -2.01671        0.52978       -0.01343
"""
Path('cyclobutene.xyz').write_text(xyz_text)
print("Written cyclobutene.xyz")

In [ ]:
import nglview as nv
from ase.io import read

mol = read('cyclobutene.xyz')
view = nv.show_ase(mol)
view.add_representation('ball+stick')
view

---
## Identifying the breaking bond

We need the atom indices of the C–C single bond opposite the C=C double bond. Run the cell below to print all C–C distances and identify it.

In [ ]:
atoms = parse_xyz('cyclobutene.xyz')
print(f"{'idx':>4}  {'elem':>4}  {'x':>8}  {'y':>8}  {'z':>8}")
for i, (el, x, y, z) in enumerate(atoms):
    print(f"{i:>4}  {el:>4}  {x:>8.4f}  {y:>8.4f}  {z:>8.4f}")

print("\nC–C distances (bonded pairs only):")
carbons = [i for i, a in enumerate(atoms) if a[0] == 'C']
for i in carbons:
    for j in carbons:
        if j > i:
            d = bond_length(atoms, i, j)
            if d < 2.0:
                print(f"  C{i}–C{j}: {d:.4f} Å")

Look at the distances above. The shortest C–C distance is the C=C double bond (~1.34 Å). The breaking bond is the single bond **not** adjacent to the double bond (~1.54 Å) — this is C0–C1.

In [ ]:
# C0–C1 is the breaking bond (1.5418 Å — the single bond opposite C=C)
IDX_C1 = 0
IDX_C4 = 1

d_eq = bond_length(atoms, IDX_C1, IDX_C4)
print(f"Breaking bond C{IDX_C1}–C{IDX_C4}: {d_eq:.4f} Å")

---
## Part 1: Building the scan geometries

We stretch the breaking bond from its equilibrium value (~1.55 Å) to 2.80 Å, keeping all other coordinates fixed. This is a **rigid scan** — a simplification, but sufficient to see the qualitative differences between methods.

In [ ]:
r_values = np.array([1.55, 1.65, 1.75, 1.85, 1.95, 2.05,
                     2.15, 2.25, 2.40, 2.55, 2.70, 2.80])

# Verify
for r in r_values[:3]:
    new_atoms = stretch_bond(atoms, IDX_C1, IDX_C4, r)
    d_check   = bond_length(new_atoms, IDX_C1, IDX_C4)
    print(f"r = {r:.2f} Å  →  actual bond = {d_check:.4f} Å")

---
## Part 2: Running the scan

We compute energies with three methods:

| Method | Notes |
|--------|-------|
| B3LYP/def2-SVP | Restricted DFT — single determinant throughout |
| CASSCF(4,4)/def2-SVP | 4 electrons in 4 orbitals (σ, σ*, π, π*) |
| NEVPT2(4,4)/def2-SVP | CASSCF + dynamic correlation |

**Active space:** The four orbitals are the σ/σ* of the breaking C–C bond and the π/π* of the C=C bond that becomes part of the diene. These four orbitals continuously transform into the four π orbitals of butadiene along the path. `ActConstraints 0` tells ORCA not to enforce active space composition — necessary for a scan where the orbital character changes.

**Starting orbitals:** The first CASSCF point uses B3LYP orbitals. Each subsequent point uses the converged orbitals from the previous point (MOREAD), which avoids convergence problems as the active space character changes along the scan.

The scan runs 12 geometries × 3 methods and takes approximately 15–20 minutes.

In [ ]:
# ── B3LYP scan ───────────────────────────────────────────────────────────
energies_b3lyp = []

for r in r_values:
    tag  = f"cb_{r:.2f}".replace('.', 'p')
    geom = atoms_to_xyz_block(stretch_bond(atoms, IDX_C1, IDX_C4, r))
    out  = run_orca(tag + '_b3lyp', f"""! B3LYP def2-SVP TightSCF
%pal nprocs {NPROCS} end
* xyz 0 1
{geom}
*
""", work_dir=work_dir)
    e = get_energy(out)
    energies_b3lyp.append(e)
    print(f"B3LYP  r = {r:.2f} Å  E = {e:.6f} Eh", flush=True)

energies_b3lyp = np.array(energies_b3lyp)
print("\n✓ B3LYP scan complete.")

In [ ]:
# ── CASSCF(4,4) + NEVPT2 scan ────────────────────────────────────────────
# Strategy:
#   Point 0  : start from B3LYP orbitals (MOREAD from b3lyp .gbw)
#   Point n>0: start from converged CASSCF orbitals of point n-1 (MOREAD)
#
# Note: ORCA is run with cwd=work_dir, so .gbw files are written there.
#       MOREAD paths inside the input file are therefore bare filenames
#       (no directory prefix needed).

energies_casscf = []
energies_nevpt2 = []
no_occ_all      = []

prev_gbw = None  # bare filename, relative to work_dir

for idx, r in enumerate(r_values):
    tag  = f"cb_{r:.2f}".replace('.', 'p')
    geom = atoms_to_xyz_block(stretch_bond(atoms, IDX_C1, IDX_C4, r))

    if prev_gbw is None:
        # First point: use B3LYP orbitals from the same geometry
        moread_gbw = f"{tag}_b3lyp.gbw"
    else:
        moread_gbw = prev_gbw

    out = run_orca(tag + '_casscf', f"""! RHF def2-SVP def2-SVP/C TightSCF NEVPT2 MOREAD
%pal nprocs {NPROCS} end
%moinp "{moread_gbw}"
%casscf
  nel          4
  norb         4
  nroots       1
  ActConstraints 0
  printlevel   3
end
* xyz 0 1
{geom}
*
""", work_dir=work_dir)

    e_cas = get_casscf_energy(out)
    e_nev = get_nevpt2_energy(out)
    energies_casscf.append(e_cas)
    energies_nevpt2.append(e_nev)
    no_occ_all.append(get_no_occupations(out))

    prev_gbw = f"{tag}_casscf.gbw"

    print(f"CASSCF r = {r:.2f} Å  "
          f"E(CAS) = {e_cas:.6f}  E(NEV) = {e_nev:.6f}", flush=True)

energies_casscf = np.array(energies_casscf)
energies_nevpt2 = np.array(energies_nevpt2)
print("\n✓ CASSCF/NEVPT2 scan complete.")

---
## Part 3: Plotting the reaction path

Energies are plotted relative to the reactant (r = 1.55 Å), converted to kcal/mol.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

methods = {
    'B3LYP':       (energies_b3lyp,  '#8e44ad', '-',  'o'),
    'CASSCF(4,4)': (energies_casscf, '#e74c3c', '--', 's'),
    'NEVPT2(4,4)': (energies_nevpt2, '#27ae60', '-',  'D'),
}

for label, (E, color, ls, marker) in methods.items():
    E_rel = (E - E[0]) * HARTREE_TO_KCAL
    ax.plot(r_values, E_rel,
            color=color, ls=ls, marker=marker, ms=6, lw=2, label=label)

ax.axhline(0, color='grey', lw=0.5, ls=':')
ax.set_xlabel('C1–C4 distance (Å)', fontsize=13)
ax.set_ylabel('Relative energy (kcal/mol)', fontsize=13)
ax.set_title('Ring opening of cyclobutene — def2-SVP', fontsize=13)
ax.legend(fontsize=12)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
plt.tight_layout()
plt.savefig('cyclobutene_pec.pdf')
plt.show()

In [ ]:
print(f"{'Method':15s}  {'Barrier (kcal/mol)':>18s}  {'TS location (Å)':>15s}")
print("-" * 55)
for label, E in [('B3LYP',       energies_b3lyp),
                 ('CASSCF(4,4)', energies_casscf),
                 ('NEVPT2(4,4)', energies_nevpt2)]:
    E_rel   = (E - E[0]) * HARTREE_TO_KCAL
    barrier = E_rel.max()
    idx_ts  = E_rel.argmax()
    print(f"{label:15s}  {barrier:>18.1f}  {r_values[idx_ts]:>15.2f}")
print(f"{'Experimental':15s}  {'~40':>18s}  {'—':>15s}")

### ✏️ Questions — Part 3

**Q1.** By how many kcal/mol does B3LYP underestimate the NEVPT2 barrier? Express this as a percentage. Is this a chemically significant error?

*Your answer:*

---

**Q2.** CASSCF(4,4) overestimates the barrier relative to NEVPT2. Why? What is missing from CASSCF that NEVPT2 adds?

*Your answer:*

---

**Q3.** This is a rigid scan — all coordinates except C0–C1 are frozen. Would a fully relaxed scan give a higher or lower barrier, and why?

*Your answer:*

---
## Part 4: Natural orbital occupations along the reaction path

Just as in Notebook 1, the CASSCF natural orbital occupation numbers tell us how much multireference character is present at each geometry. The four active orbitals are σ, σ*, π, π* — at the TS we expect the σ and σ* occupations to approach 1.0, indicating a diradical.

In [ ]:
no_array = np.array([occs[:4] if len(occs) >= 4 else [float('nan')]*4
                     for occs in no_occ_all])

fig, ax = plt.subplots(figsize=(8, 4))

colors = ['#2980b9', '#27ae60', '#e67e22', '#c0392b']
labels = ['NO 1 (σ / π₁)', 'NO 2 (π / π₂)', 'NO 3 (π* / π₃*)', 'NO 4 (σ* / π₄*)']

for i in range(4):
    ax.plot(r_values, no_array[:, i],
            'o-', color=colors[i], lw=2, label=labels[i])

ax.axhline(1.0, color='grey', lw=0.8, ls=':', label='occ = 1.0')
ax.set_xlabel('C1–C4 distance (Å)', fontsize=13)
ax.set_ylabel('NO occupation number', fontsize=13)
ax.set_title('Active space NO occupations — CASSCF(4,4)/def2-SVP', fontsize=13)
ax.set_ylim(-0.05, 2.05)
ax.legend(fontsize=10, loc='center right')
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
plt.tight_layout()
plt.savefig('cyclobutene_no_occ.pdf')
plt.show()

### ✏️ Questions — Part 4

**Q4.** At the TS geometry, what are the occupation numbers of the four active orbitals? What does this tell you about the electronic character of the TS?

*Your answer:*

---

**Q5.** At r = 1.55 Å (reactant), the σ occupation should be close to 2.0 and σ* close to 0.0. Is this what you see? What does this mean for the multireference character of cyclobutene at equilibrium?

*Your answer:*

---

**Q6.** Compare the r value where the NO occupations cross 1.0 with the r value of the barrier maximum. Are they the same? What does this tell you?

*Your answer:*

---

**Q7.** B3LYP uses a single determinant with doubly-occupied orbitals. Given what you see in the NO occupation plot, explain *why* B3LYP must fail near the TS — not just that it does, but the fundamental reason.

*Your answer:*

---
## Part 5: When should you worry?

The cyclobutene example illustrates a general principle: any reaction where a σ bond breaks homolytically will have a TS with diradical character, and a single-determinant method will underestimate the barrier.

**A practical diagnostic:** run a CASSCF(2,2) or CASSCF(4,4) calculation at the suspected TS geometry and inspect the NO occupations. If the two frontier orbitals have occupations significantly away from (2.0, 0.0) — say, both between 1.2 and 0.8 — multireference treatment is essential.

Reactions where this matters in practice:

- Electrocyclic reactions (especially thermally forbidden pathways)
- Retro-Diels-Alder reactions
- Radical recombination and β-scission
- Transition metal catalysis with open-shell intermediates
- Photochemical reactions and conical intersections

### ✏️ Final question

**Q8.** Think about a reaction relevant to your own research or field. Could it involve a TS with diradical character? What evidence would you look for before deciding whether to use a multireference method?

*Your answer:*

---
## Summary

Fill in the blanks:

1. The active space for cyclobutene ring opening is CASSCF(**___**,**___**) — the σ/σ* of the breaking bond and the π/π* of the forming diene.

2. At the TS, the two frontier NO occupations are approximately **_____** and **_____** — indicating **strong / weak** (circle one) multireference character.

3. B3LYP **underestimates / overestimates** (circle one) the barrier by approximately **_____** kcal/mol relative to NEVPT2.

4. CASSCF **underestimates / overestimates** (circle one) the barrier relative to NEVPT2 because it lacks **_____** correlation.

5. The correct level of theory for this problem is **_____**, which captures both static and dynamic correlation.

6. A practical diagnostic for multireference character at a TS is to inspect the **_____** from a CASSCF calculation and check whether frontier orbital occupations deviate significantly from **_____** and **_____**.

---

*End of Multireference Quantum Chemistry exercises.*

For further reading see the ORCA manual sections on CASSCF and NEVPT2, and the review by Roos et al. on multiconfigurational quantum chemistry.